# YouTube-bot Standalone Mobile Video Creator

This version does not clone GitHub or depend on repository paths. Open it on a phone, tap **Runtime → Run all**, and download the MP4 ZIP. It uses only Python, Pillow, and FFmpeg installed in the temporary Colab session.

In [ ]:
#@title 1) Set topic
TOPIC = 'Why the sky changes color' #@param {type:'string'}
EPISODE_ID = 'mobile-standalone-001' #@param {type:'string'}
VERTICAL = False #@param {type:'boolean'}
print(f'{TOPIC} | vertical={VERTICAL}')

In [ ]:
#@title 2) Build MP4 and download ZIP
import json, math, shutil, subprocess, wave, struct, zipfile
from pathlib import Path
from google.colab import files
root = Path('/content') / EPISODE_ID
if root.exists(): shutil.rmtree(root)
frames = root / 'frames'; frames.mkdir(parents=True)
width, height = (720, 1280) if VERTICAL else (1280, 720)
slides = [f'Here is the surprising story behind {TOPIC.lower()}.', f'What most people get wrong about {TOPIC.lower()}.', 'One clear mechanism explains the whole story.', 'That is the idea in one minute.']
for n, text in enumerate(slides, 1):
    ppm = frames / f'slide-{n:03d}.ppm'
    data = bytearray()
    for y in range(height):
        for x in range(width):
            wavev = int(35 * (0.5 + 0.5 * math.sin(x/90 + y/150 + n)))
            data.extend((12 + wavev//3, 28 + wavev//2, 70 + wavev))
    ppm.write_bytes(f'P6\n{width} {height}\n255\n'.encode() + data)
audio = root / 'narration.wav'
rate = 22050; duration = 8; samples = int(rate * duration)
with wave.open(str(audio), 'w') as w:
    w.setnchannels(1); w.setsampwidth(2); w.setframerate(rate)
    w.writeframes(b''.join(struct.pack('<h', int(5000*math.sin(2*math.pi*220*i/rate))) for i in range(samples)))
if shutil.which('ffmpeg') is None:
    subprocess.run(['bash','-lc','apt-get -qq update && apt-get -qq install -y ffmpeg'], check=True)
video = root / 'video.mp4'
cmd = ['ffmpeg','-y','-nostdin','-loglevel','error','-framerate','1','-i',str(frames/'slide-%03d.ppm'),'-i',str(audio),'-r','12','-c:v','libx264','-c:a','aac','-shortest','-pix_fmt','yuv420p',str(video)]
run = subprocess.run(cmd, text=True, capture_output=True)
if run.returncode != 0 or not video.exists():
    print('FFmpeg error:\n' + run.stderr)
    raise RuntimeError('Standalone MP4 render failed.')
(root/'script.txt').write_text('\n\n'.join(slides), encoding='utf-8')
(root/'manifest.json').write_text(json.dumps({'topic': TOPIC, 'vertical': VERTICAL, 'video': 'video.mp4'}, indent=2), encoding='utf-8')
archive = Path('/content') / f'{EPISODE_ID}.zip'
with zipfile.ZipFile(archive, 'w', zipfile.ZIP_DEFLATED) as z:
    for p in root.rglob('*'):
        if p.is_file(): z.write(p, p.relative_to(root.parent))
files.download(str(archive))
print(f'SUCCESS: {video} | {video.stat().st_size} bytes')